In [2]:
# ============================================================
# CELL PURPOSE: Load engineered features
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd

DATA_PATH = "/Users/sanjana/Desktop/Hype-Predictor/Data"

df = pd.read_csv(f"{DATA_PATH}/clean_data.csv")

# Recreate scaled version (or reuse if saved)
cols = ["trends_score", "reddit_score", "youtube_score", "news_score"]

df_z = df.copy()
for col in cols:
    df_z[col] = (df[col] - df[col].mean()) / df[col].std()

df_scaled = df_z.copy()
for col in cols:
    df_scaled[col] = 100 * (df_z[col] - df_z[col].min()) / (df_z[col].max() - df_z[col].min())

print(df_scaled)

           product  trends_score  reddit_score  youtube_score  news_score
0    Air Jordan 11    100.000000      0.000000       8.192101   94.520548
1  Nvidia RTX 5090     17.469800     11.596741     100.000000  100.000000
2    Owala FreeSip     25.368379      9.086708       0.000000    0.000000
3          PS5 Pro     20.390283    100.000000      13.899106   97.260274
4        iPhone 17      0.000000     31.059224      50.309085   98.630137


In [4]:
# ============================================================
# CELL PURPOSE: Compute final hype score
# ============================================================

weights = {
    "trends_score": 0.25,
    "reddit_score": 0.20,
    "youtube_score": 0.30,
    "news_score": 0.25
}

df_final = df_scaled.copy()

df_final["hype_score"] = (
    df_scaled["trends_score"] * weights["trends_score"] +
    df_scaled["reddit_score"] * weights["reddit_score"] +
    df_scaled["youtube_score"] * weights["youtube_score"] +
    df_scaled["news_score"] * weights["news_score"]
)

print(df_final[["product", "hype_score"]])

           product  hype_score
0    Air Jordan 11   51.087767
1  Nvidia RTX 5090   61.686798
2    Owala FreeSip    8.159436
3          PS5 Pro   53.582371
4        iPhone 17   45.962105


In [6]:
# ============================================================
# CELL PURPOSE: Measure consistency across signals
# ============================================================

cols = ["trends_score", "reddit_score", "youtube_score", "news_score"]

# Standard deviation across signals
df_final["consistency_std"] = df_scaled[cols].std(axis=1)

print(df_final[["product", "consistency_std"]])

           product  consistency_std
0    Air Jordan 11        53.938669
1  Nvidia RTX 5090        49.402457
2    Owala FreeSip        11.962924
3          PS5 Pro        47.133492
4        iPhone 17        41.326840


In [8]:
# ============================================================
# Normalize consistency (invert: lower std = better)
# ============================================================

min_val = df_final["consistency_std"].min()
max_val = df_final["consistency_std"].max()

df_final["consistency_score"] = 100 * (
    1 - (df_final["consistency_std"] - min_val) / (max_val - min_val)
)

print(df_final[["product", "consistency_score"]])

           product  consistency_score
0    Air Jordan 11           0.000000
1  Nvidia RTX 5090          10.806745
2    Owala FreeSip         100.000000
3          PS5 Pro          16.212165
4        iPhone 17          30.045514


In [10]:
# ============================================================
# Combine original hype score with consistency
# ============================================================

consistency_weight = 0.10  # small influence

df_final["hype_score_adjusted"] = (
    df_final["hype_score"] * (1 - consistency_weight) +
    df_final["consistency_score"] * consistency_weight
)

print(df_final[["product", "hype_score", "consistency_score", "hype_score_adjusted"]])

           product  hype_score  consistency_score  hype_score_adjusted
0    Air Jordan 11   51.087767           0.000000            45.978991
1  Nvidia RTX 5090   61.686798          10.806745            56.598793
2    Owala FreeSip    8.159436         100.000000            17.343493
3          PS5 Pro   53.582371          16.212165            49.845350
4        iPhone 17   45.962105          30.045514            44.370446


In [12]:
df_final = df_final.sort_values(by="hype_score_adjusted", ascending=False)

print(df_final[["product", "hype_score_adjusted"]])

           product  hype_score_adjusted
1  Nvidia RTX 5090            56.598793
3          PS5 Pro            49.845350
0    Air Jordan 11            45.978991
4        iPhone 17            44.370446
2    Owala FreeSip            17.343493


In [ ]:
"""🥇 Nvidia
Strong YouTube + News
👉 Media-driven hype → makes sense at #1
🥈 PS5
Massive Reddit discussion
👉 Community-driven hype → strong #2
🥉 Air Jordan
High Trends but weak elsewhere
👉 Slightly penalized → correct
4️⃣ iPhone
Balanced but no spikes
👉 Stable but not dominant → correct
❌ Owala
Low everywhere
👉 Even with consistency boost → still low → correct"""